# LAV-DF private, authenticated baseline workflow

**Run setup and diagnostics in order. Do not train until both the environment and dataset audits pass.** Review provider terms and confirm that Colab and your private storage are permitted. Never put tokens, videos, features, identity metadata, reports or weights in public Git.

The Colab control kernel may show Python 3.13: this is expected. Model loading, extraction, dataset processing, training and evaluation run in a separate Python 3.12.10 virtual environment at `/content/lavdf-cuda-env/bin/python`. Its Python installation is under `/content/lavdf-python312-bootstrap`. The next diagnostic prints both interpreters and runs a CUDA calculation in the project environment. Select a T4 GPU and use a fresh Linux x86-64 Colab session with sufficient free storage and internet access for the setup downloads.

If setup is interrupted, rerun the setup cell: verified files and the virtual environment are reused, and package installation resumes. If it reports an unrecognized existing repository or environment, preserve any needed files and start a fresh Colab runtime; the setup intentionally does not delete unknown files. If a download or installation fails, check the error and connection, then rerun the cell. Private checkpoints and Hugging Face authentication are only needed in later cells.

After mounting Drive, the notebook stages the trusted DINOv2 source checkout under `/content` and verifies its pinned commit and clean working tree. It repeats this automatically after a runtime reset; inference uses the verified local path. If an existing local copy fails verification, preserve and inspect it rather than bypassing checks.


In [1]:
GPU_DIAGNOSTIC_PASSED = MODEL_SMOKE_PASSED = DATASET_VERIFIED = False
import hashlib, os, re, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

# Colab can set Python-specific paths for its control kernel; do not pass them to Python 3.12.
PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO_COMMIT = 'f932ba021b9c078379130e2fe2abbb66539cafe9'  # paste the exact reviewed 40-character commit from the instructions
REPO = Path('/content/deepfake-detection-dtw')
BOOTSTRAP = Path('/content/lavdf-python312-bootstrap')
VENV = Path('/content/lavdf-cuda-env')
PYTHON = str(VENV / 'bin/python')
PYTHON_SHA256 = '3b7de567f4a8a1c3e5be2d51af3023dd3c1075cfacb6cd8efb511e0ef4b31fa8'
PYTHON_URL = 'https://github.com/astral-sh/python-build-standalone/releases/download/20250529/cpython-3.12.10%2B20250529-x86_64-unknown-linux-gnu-install_only.tar.gz'
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT), 'Set REPO_COMMIT to the reviewed commit SHA'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Pradeepbeleri/deepfake-detection-dtw.git', str(REPO)], check=True)
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=all'],text=True).strip(), 'Repository has local changes; preserve them and use a clean checkout'
subprocess.run(['git','-C',str(REPO),'fetch','origin',REPO_COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'switch','--detach',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == REPO_COMMIT

# Refuse to reuse an unrelated or partially created installation. Never overwrite it.
marker = BOOTSTRAP / '.verified-sha256'
if not BOOTSTRAP.exists():
    with tempfile.TemporaryDirectory(prefix='lavdf-python-', dir=BOOTSTRAP.parent) as temp:
        archive = Path(temp) / 'python.tar.gz'
        urllib.request.urlretrieve(PYTHON_URL, archive)
        with archive.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != PYTHON_SHA256:
            raise RuntimeError('Python archive SHA-256 mismatch; no archive was used')
        # Extract and mark in a temporary sibling: interruption cannot leave a
        # partially extracted installation at the final path.
        staged = Path(temp) / 'bootstrap'
        staged.mkdir(mode=0o700)
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(path=staged, filter='data')
        (staged / '.verified-sha256').write_text(PYTHON_SHA256)
        staged.rename(BOOTSTRAP)
assert marker.is_file() and marker.read_text().strip() == PYTHON_SHA256, 'Existing bootstrap is incomplete or unrecognized. Preserve your files and start a fresh Colab runtime.'
standalone = BOOTSTRAP / 'python/bin/python3.12'
assert subprocess.check_output([str(standalone),'-c','import sys; print(".".join(map(str,sys.version_info[:3])))'],text=True,env=PROJECT_ENV).strip() == '3.12.10'
venv_marker = VENV / '.lavdf-python-sha256'
if not VENV.exists():
    # Mark our directory before provisioning so an interrupted venv can be resumed.
    VENV.mkdir(mode=0o700)
    venv_marker.write_text(PYTHON_SHA256)
assert venv_marker.is_file() and venv_marker.read_text().strip() == PYTHON_SHA256, 'Existing virtual environment is unrecognized. Preserve your files and start a fresh Colab runtime.'
venv_complete = VENV / '.venv-complete'
if not venv_complete.is_file() or not Path(PYTHON).is_file():
    subprocess.run([str(standalone),'-m','venv',str(VENV)],env=PROJECT_ENV,check=True)
    venv_complete.write_text('ok')
print('Project interpreter:', PYTHON, flush=True)

# GPU wheels and CUDA libraries are confined to this virtual environment.
cuda_packages = ['nvidia-cuda-nvrtc-cu12','nvidia-cuda-runtime-cu12','nvidia-cuda-cupti-cu12','nvidia-cudnn-cu12','nvidia-cublas-cu12','nvidia-cufft-cu12','nvidia-curand-cu12','nvidia-cusolver-cu12','nvidia-cusparse-cu12','nvidia-nccl-cu12','nvidia-nvtx-cu12','nvidia-nvjitlink-cu12','triton']
subprocess.run([PYTHON,'-m','pip','install','--index-url','https://pypi.org/simple','-c',str(REPO/'requirements-colab-cu124.txt'),*cuda_packages],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','--no-deps','--index-url','https://download.pytorch.org/whl/cu124','torch==2.5.1+cu124','torchvision==0.20.1+cu124'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','install','-c',str(REPO/'requirements-colab-cu124.txt'),'-e',str(REPO),'huggingface_hub==0.30.2'],env=PROJECT_ENV,check=True)
subprocess.run([PYTHON,'-m','pip','check'],env=PROJECT_ENV,check=True)
print('Setup completed. Run the next cell to check both interpreters and CUDA.', flush=True)


Project interpreter: /content/lavdf-cuda-env/bin/python
Setup completed. Run the next cell to check both interpreters and CUDA.


In [5]:
import os

os.environ["MPLBACKEND"] = "Agg"

if "PROJECT_ENV" in globals():
    PROJECT_ENV["MPLBACKEND"] = "Agg"

print("Matplotlib backend:", os.environ["MPLBACKEND"])

Matplotlib backend: Agg


In [6]:
GPU_DIAGNOSTIC_PASSED = False
import os, platform, subprocess, sys
from pathlib import Path

print('=== Colab control kernel (not used for project ML) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('Python 3.13 here is OK; the project uses the separate interpreter below.', flush=True)

PYTHON = '/content/lavdf-cuda-env/bin/python'
if not Path(PYTHON).is_file():
    raise RuntimeError('Project interpreter not found. Run the setup cell above and resolve any error before retrying.')

DIAGNOSTIC = r"""
import importlib.metadata as md, platform, sys
import torch, torchvision, whisper, mediapipe
import deepfake_dtw.encoders
print('\n=== Project ML environment (used by project commands) ===', flush=True)
print('Python version:', platform.python_version(), flush=True)
print('Python executable:', sys.executable, flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('torchvision:', torchvision.__version__, flush=True)
print('PyTorch CUDA build:', torch.version.cuda, flush=True)
print('Whisper:', md.version('openai-whisper'), '| MediaPipe:', mediapipe.__version__, flush=True)
available = torch.cuda.is_available()
print('CUDA available:', available, flush=True)
print('GPU name:', torch.cuda.get_device_name(0) if available else 'Unavailable', flush=True)
if sys.version_info[:3] != (3, 12, 10):
    raise RuntimeError('Unexpected project Python version; rerun setup or use a fresh runtime.')
if torch.__version__ != '2.5.1+cu124' or torchvision.__version__ != '0.20.1+cu124' or torch.version.cuda != '12.4':
    raise RuntimeError('Unexpected PyTorch CUDA build; rerun setup and check pip errors.')
if not available:
    raise RuntimeError('CUDA unavailable in the project interpreter. Select a T4 GPU in Runtime > Change runtime type, reconnect, rerun setup and check the runtime driver.')
x = torch.tensor([1.0, 2.0], device='cuda')
if (x*x).sum().item() != 5.0:
    raise RuntimeError('CUDA tensor calculation returned an unexpected result.')
torch.cuda.synchronize()
print('CUDA tensor operation: PASSED', flush=True)
"""
result = subprocess.run(
    [PYTHON, '-c', DIAGNOSTIC],
    env={
        k: v for k, v in os.environ.items()
        if k not in ('PYTHONPATH', 'PYTHONHOME')
    },
    capture_output=True,
    text=True
)

print("=== STDOUT ===")
print(result.stdout)

print("=== STDERR ===")
print(result.stderr)

print("Exit code:", result.returncode)

GPU_DIAGNOSTIC_PASSED = result.returncode == 0

=== Colab control kernel (not used for project ML) ===
Python version: 3.13.16
Python executable: /usr/bin/python3
Python 3.13 here is OK; the project uses the separate interpreter below.
=== STDOUT ===

=== Project ML environment (used by project commands) ===
Python version: 3.12.10
Python executable: /content/lavdf-cuda-env/bin/python
PyTorch: 2.5.1+cu124
torchvision: 0.20.1+cu124
PyTorch CUDA build: 12.4
Whisper: 20250625 | MediaPipe: 0.10.21
CUDA available: True
GPU name: Tesla T4
CUDA tensor operation: PASSED

=== STDERR ===

Exit code: 0


In [7]:
import os, json, subprocess, sys
from pathlib import Path

PROJECT_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME')}

REPO = Path('/content/deepfake-detection-dtw')
PYTHON = '/content/lavdf-cuda-env/bin/python'
# Mount only the Colab control kernel's Drive integration; ML uses the isolated interpreter.
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
PRIVATE = Path('/content/drive/MyDrive/deepfake-private')  # only if provider/institution authorize this storage
MEDIA = PRIVATE / 'lavdf' / 'media'
FEATURES = PRIVATE / 'lavdf' / 'features'
REPORTS = PRIVATE / 'lavdf' / 'reports'
PROVENANCE = PRIVATE / 'lavdf' / 'curated-provenance.json'
PLAN = PRIVATE / 'lavdf' / 'plan-source-groups.json'  # change this path for a separate experiment
METADATA_LOCAL = PRIVATE / 'lavdf' / 'metadata.min.json'
WHISPER = PRIVATE / 'tiny.en.pt'
DINO = PRIVATE / 'dinov2_vits14_pretrain.pth'
DINO_DRIVE_SOURCE = PRIVATE / 'dinov2-src'  # trusted self-contained checkout in private Drive
DINO_SOURCE = Path('/content/dinov2-src-7764ea0f')  # fast local checkout; verified before use
DINO_SHA256 = 'b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9'  # user-supplied transfer hash; verify your file provenance
assert (REPO / 'src/deepfake_dtw/lavdf.py').exists() and Path(PYTHON).is_file(), 'Complete repository environment setup first'
assert PRIVATE.is_dir(), 'Mount and create your authorized private storage root before continuing'
assert len(DINO_SHA256) == 64 and WHISPER.is_file() and DINO.is_file(), 'Provide and verify both permitted model files'
# Copy to local Colab disk and verify the pinned commit and clean working tree.
# Reuse is allowed only after verification; no existing checkout is overwritten.
subprocess.run([PYTHON, '-m', 'deepfake_dtw.stage_dinov2', '--source', str(DINO_DRIVE_SOURCE), '--destination', str(DINO_SOURCE)], cwd=REPO, env=PROJECT_ENV, check=True)
for directory in (MEDIA, FEATURES, REPORTS):
    directory.mkdir(parents=True, mode=0o700, exist_ok=True)


Mounted at /content/drive


In [8]:
from pathlib import Path

items = {
    "Dataset plan": PRIVATE / "lavdf/plan-source-groups.json",
    "Metadata": PRIVATE / "lavdf/metadata.min.json",
    "Whisper checkpoint": WHISPER,
    "DINOv2 checkpoint": DINO,
    "Archive in Drive": PRIVATE / "lavdf/archive/LAV-DF.tar",
    "Archive in Colab": Path("/content/lavdf-archive/LAV-DF.tar"),
}

for name, path in items.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

print("Saved feature files:", len(list(FEATURES.glob("*.npz"))))

Dataset plan: FOUND
Metadata: FOUND
Whisper checkpoint: FOUND
DINOv2 checkpoint: FOUND
Archive in Drive: MISSING
Archive in Colab: MISSING
Saved feature files: 20


In [9]:
MODEL_SMOKE_PASSED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False), 'Run the GPU diagnostic successfully first'
# Required model-loading check. It performs no classifier training.
MODEL_SMOKE = "import json, numpy as np, sys, torch\nfrom deepfake_dtw.encoders import WhisperAudioEncoder, DinoVisualEncoder\nfrom deepfake_dtw.preprocessing import prepare_audio, prepare_video\ndevice=torch.device('cuda')\na=WhisperAudioEncoder(device=device,mode='pretrained',checkpoint=sys.argv[1])\nv=DinoVisualEncoder(device=device,checkpoint=sys.argv[2],sha256=sys.argv[3],source_dir=sys.argv[4])\naudio=prepare_audio(np.zeros(16000,dtype=np.float32),16000)\nvideo=prepare_video(np.zeros((1,64,64,3),dtype=np.uint8),[0.0])\naf=a.extract(audio); vf=v.extract(video)\nassert torch.isfinite(af.values).all() and torch.isfinite(vf.values).all()\nassert af.values.shape==(1,1500,384) and vf.values.shape==(1,384)\ntorch.cuda.synchronize()\nprint(json.dumps({'model_interpreter':sys.executable,'device':str(device),'gpu':torch.cuda.get_device_name(0),'audio_shape':list(af.values.shape),'video_shape':list(vf.values.shape),'audio_parameters':sum(p.numel() for p in a.model.parameters()),'visual_parameters':sum(p.numel() for p in v.model.parameters()),'gpu_peak_bytes':torch.cuda.max_memory_allocated()},indent=2))\n"
model_env = dict(PROJECT_ENV, XFORMERS_DISABLED='1', OMP_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8')
subprocess.run([PYTHON,'-c',MODEL_SMOKE,str(WHISPER),str(DINO),DINO_SHA256,str(DINO_SOURCE)],env=model_env,check=True)
MODEL_SMOKE_PASSED = True


In [10]:
print("MODEL_SMOKE_PASSED:", MODEL_SMOKE_PASSED)

MODEL_SMOKE_PASSED: True


## Authorized archive and private storage

The provider supplies `LAV-DF.tar`, not individual media files. Keep its 25.61 GB archive and Hugging Face cache in authorized private Drive storage so they survive runtime resets. Check Drive capacity for the archive, selected videos, features and temporary download files. Hugging Face resumes incomplete downloads where supported; a reset can still require retrying. Authentication is interactive and the token is never placed in code or Git. Use only after reviewing provider and underlying dataset terms.


In [11]:
import shutil

disk = shutil.disk_usage("/content")
print(f"Available disk space: {disk.free / 1e9:.2f} GB")

Available disk space: 110.23 GB


In [12]:
import os
import subprocess
from pathlib import Path

ARCHIVE_DIR = Path("/content/lavdf-archive")
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE = ARCHIVE_DIR / "LAV-DF.tar"

ARCHIVE_SHA256 = (
    "2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4"
)

HF_HOME = PRIVATE / "lavdf" / "hf-private"

env = dict(PROJECT_ENV)
env["HF_HOME"] = str(HF_HOME)
env["MPLBACKEND"] = "Agg"

script = """
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="ControlNet/LAV-DF",
    repo_type="dataset",
    filename="LAV-DF.tar",
    local_dir="/content/lavdf-archive",
)

print("Archive downloaded:", path, flush=True)
"""

subprocess.run(
    [PYTHON, "-c", script],
    cwd=REPO,
    env=env,
    check=True
)

print("Archive found:", ARCHIVE.is_file())

Archive found: True


In [13]:
import hashlib
from pathlib import Path

ARCHIVE = Path("/content/lavdf-archive/LAV-DF.tar")

EXPECTED_SHA256 = (
    "2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4"
)

print(f"Archive size: {ARCHIVE.stat().st_size / 1e9:.2f} GB", flush=True)

sha256 = hashlib.sha256()

with ARCHIVE.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

actual = sha256.hexdigest()

print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256:  ", actual)

ARCHIVE_VERIFIED = actual == EXPECTED_SHA256
print("ARCHIVE_VERIFIED:", ARCHIVE_VERIFIED)

assert ARCHIVE_VERIFIED, "Archive integrity verification failed!"

Archive size: 25.61 GB
Expected SHA256: 2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4
Actual SHA256:   2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4
ARCHIVE_VERIFIED: True


In [15]:
from pathlib import Path

ARCHIVE = Path("/content/lavdf-archive/LAV-DF.tar")

ARCHIVE_SHA256 = (
    "2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4"
)

assert ARCHIVE.is_file(), "Local LAV-DF archive is missing"
assert globals().get("ARCHIVE_VERIFIED", False), "Verify archive SHA-256 first"

print("Archive ready:", ARCHIVE)
print("Integrity verified:", ARCHIVE_VERIFIED)


Archive ready: /content/lavdf-archive/LAV-DF.tar
Integrity verified: True


## Plan an explicitly scoped source-group baseline

The default `source-group-only` mode uses provider original/source links and official partitions. It makes **no identity-disjointness claim**: different recordings of one participant or undisclosed audio sources may cross partitions. The importer excludes untraceable or cross-partition source links rather than inventing identities, and stops if it cannot construct the requested partitions. The provider and upstream dataset conditions continue to apply. The existing `identity` mode remains available if you have a complete audited sidecar; set `PLAN_MODE` accordingly and use a separate plan path.


In [16]:
# Select scope explicitly; the default does not require an identity sidecar.
import hashlib
from collections import Counter
PLAN_MODE = 'source-group-only'  # or 'identity' if a complete audited identity sidecar is available
USAGE_RIGHTS = 'User reports authorized noncommercial academic research; provider and VoxCeleb2 conditions apply.'
assert PLAN_MODE in ('source-group-only', 'identity')
assert METADATA_LOCAL.is_file(), f'Provide the provider metadata at {METADATA_LOCAL}'
PLAN_STAMP = PLAN.with_suffix('.inputs.json')
def digest_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
inputs = {'metadata_sha256': digest_file(METADATA_LOCAL), 'mode': PLAN_MODE,
          'limit_per_split': 60, 'seed': 42, 'usage_rights': USAGE_RIGHTS,
          'repository_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()}
command = [PYTHON, '-m', 'deepfake_dtw.lavdf', '--metadata', str(METADATA_LOCAL),
           '--media-root', str(MEDIA), '--feature-root', str(FEATURES), '--output', str(PLAN),
           '--limit-per-split', '60', '--seed', '42']
if PLAN_MODE == 'source-group-only':
    command += ['--source-group-only', '--usage-rights', USAGE_RIGHTS]
else:
    assert PROVENANCE.is_file(), f'Identity mode needs an audited sidecar: {PROVENANCE}'
    inputs['provenance_sha256'] = digest_file(PROVENANCE)
    command += ['--provenance', str(PROVENANCE)]
if PLAN.exists():
    assert PLAN_STAMP.is_file(), 'Existing plan has no input stamp; preserve it and choose a new PLAN path.'
    prior_inputs = json.loads(PLAN_STAMP.read_text())
    assert {k:v for k,v in prior_inputs.items() if k != 'repository_commit'} == {k:v for k,v in inputs.items() if k != 'repository_commit'}, 'Existing plan inputs differ; preserve it and choose a new PLAN path.'
    if prior_inputs.get('repository_commit') != inputs['repository_commit']:
        assert PLAN_MODE == 'source-group-only', 'Identity-sidecar plans require review when changing code versions.'
        verify_initial = '''import json,sys
from deepfake_dtw.lavdf_recovery import verify_initial_plan
with open(sys.argv[2]) as f: metadata=json.load(f)
verify_initial_plan(sys.argv[1],metadata,sys.argv[3],sys.argv[4],sys.argv[5],sys.argv[6],limit=60,seed=42)
'''
        subprocess.run([PYTHON, '-c', verify_initial, str(PLAN), str(METADATA_LOCAL), str(REPORTS/'replacement-journal'), str(MEDIA), str(FEATURES), USAGE_RIGHTS], cwd=REPO, env=PROJECT_ENV, check=True)
        print('Verified the original deterministic plan from the earlier code revision; original stamp retained.')
    else:
        print('Reusing existing plan with matching input hashes.')
else:
    assert not PLAN_STAMP.exists(), 'Plan stamp exists without plan; preserve it and choose a new PLAN path.'
    subprocess.run(command, cwd=REPO, env=PROJECT_ENV, check=True)
    with PLAN_STAMP.open('x') as stream:
        json.dump(inputs, stream, indent=2)
records = json.loads(PLAN.read_text())
print('Scope:', PLAN_MODE, '(identity disjointness is not established)' if PLAN_MODE == 'source-group-only' else '(conditional on the audited sidecar)')
print('Planned split, label and manipulation counts:', dict(Counter((r['partition'], r['label'], r['manipulation']) for r in records)))


Verified the original deterministic plan from the earlier code revision; original stamp retained.
Scope: source-group-only (identity disjointness is not established)
Planned split, label and manipulation counts: {('train', 'real', 'real'): 30, ('train', 'fake', 'both_modified'): 11, ('train', 'fake', 'video_modified'): 9, ('train', 'fake', 'audio_modified'): 10, ('validation', 'real', 'real'): 30, ('validation', 'fake', 'audio_modified'): 11, ('validation', 'fake', 'video_modified'): 10, ('validation', 'fake', 'both_modified'): 9, ('test', 'real', 'real'): 30, ('test', 'fake', 'video_modified'): 12, ('test', 'fake', 'both_modified'): 10, ('test', 'fake', 'audio_modified'): 8}


## Verify archive and extract the selected videos

The full archive hash is checked before extraction. Only the explicitly planned regular MP4 members are copied; paths and existing files are validated. Rerunning reuses existing files only after comparing them with the archive. A mismatch stops for manual inspection; nothing is overwritten. Hashing a 25.61 GB file takes time, especially on Drive.


In [17]:
subprocess.run([PYTHON, '-m', 'deepfake_dtw.lavdf_archive', '--archive', str(ARCHIVE), '--plan', str(PLAN), '--media-root', str(MEDIA), '--sha256', ARCHIVE_SHA256], cwd=REPO, env=PROJECT_ENV, check=True)


CompletedProcess(args=['/content/lavdf-cuda-env/bin/python', '-m', 'deepfake_dtw.lavdf_archive', '--archive', '/content/lavdf-archive/LAV-DF.tar', '--plan', '/content/drive/MyDrive/deepfake-private/lavdf/plan-source-groups.json', '--media-root', '/content/drive/MyDrive/deepfake-private/lavdf/media', '--sha256', '2abc0646cf1469f0bebec60a792fec8b6d670cd9e6133105139110c4316b20a4'], returncode=0)

## Extract and audit before training

Verified feature/report pairs are reused. Only complete, source-hash-matched reports with a predefined `face_unusable` reason can trigger deterministic replacement; infrastructure and other failures stop. Replacement maintains official split, real/fake balance and exact source pairing; same manipulation category is preferred. Reports and the replayable replacement journal stay private. A partial output is preserved in the private `interrupted` directory before retry. Plans are frozen once training or test artifacts exist. Quality decisions do not use predictions or evaluation results.


In [18]:
print("Archive:", ARCHIVE)
print("GPU:", GPU_DIAGNOSTIC_PASSED)
print("Models:", MODEL_SMOKE_PASSED)

Archive: /content/lavdf-archive/LAV-DF.tar
GPU: True
Models: True


In [19]:
DATASET_VERIFIED = False
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False), 'Run both GPU checks first'
assert PLAN_MODE == 'source-group-only', 'Automatic replacement requires provider source-group mode; preserve identity-sidecar results and review failures manually.'
env = dict(PROJECT_ENV, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', XFORMERS_DISABLED='1', CUBLAS_WORKSPACE_CONFIG=':4096:8')
freeze = [PRIVATE/'lavdf'/'baseline.pt', PRIVATE/'lavdf'/'baseline.progress.pt', REPORTS/'training.json', REPORTS/'held-out-test.json']
command = [PYTHON, '-m', 'deepfake_dtw.lavdf_recovery', '--plan', str(PLAN), '--metadata', str(METADATA_LOCAL),
           '--archive', str(ARCHIVE), '--archive-sha256', ARCHIVE_SHA256, '--media-root', str(MEDIA), '--feature-root', str(FEATURES),
           '--reports', str(REPORTS), '--journal', str(REPORTS/'replacement-journal'), '--audio-checkpoint', str(WHISPER),
           '--visual-checkpoint', str(DINO), '--visual-sha256', DINO_SHA256, '--dinov2-source', str(DINO_SOURCE), '--seed', '42']
for artifact in freeze:
    command += ['--freeze-check', str(artifact)]
subprocess.run(command, cwd=REPO, env=env, check=True)
records = json.loads(PLAN.read_text())
print('Feature extraction and predefined face-quality recovery complete; inspect the private replacement journal and quality reports.')


Feature extraction and predefined face-quality recovery complete; inspect the private replacement journal and quality reports.


## Optional controlled timing audit

This runs measured ±0.2-second audio-timestamp perturbations on selected genuine training and validation clips only; the test partition remains untouched. It checks candidate audio-energy/lip-aperture responses, but cannot certify correspondence or support a detector ablation by itself. Review the private report and its failures. No DTW ablation or improvement claim is produced by this cell.


In [ ]:
# Exploratory timing audit; uses real selected videos, never the held-out test partition.
DTW_AUDIT = REPORTS / 'timing-audit.json'
if not DTW_AUDIT.exists():
    subprocess.run([PYTHON, '-m', 'deepfake_dtw.dtw_validation', str(PLAN), str(DTW_AUDIT), '--limit-per-partition', '20', '--seed', '42'], cwd=REPO, env=env, check=True)
else:
    saved_audit = json.loads(DTW_AUDIT.read_text())
    assert saved_audit.get('manifest_sha256') == digest_file(PLAN), 'Existing timing audit belongs to a different plan; choose a new report path.'
    by_id = {r['sample_id']: r for r in records}
    for section in saved_audit['partitions'].values():
        for sample in section['samples']:
            assert digest_file(Path(by_id[sample['sample_id']]['video_path'])) == sample['video_sha256'], 'Timing audit source video changed.'
    print('Reusing matching private timing audit; this does not establish DTW validity.')


## Train and final evaluation

Review the private quality reports, labels, manipulation counts, original links and partition audit before training. The CUDA trainer fits normalization on training data only and uses validation for checkpoint and threshold selection. Source-group-only evaluation cannot establish unseen-identity performance. The test partition must not guide model or threshold choices. Outputs remain private; sigmoid scores are not calibrated probabilities.

The trainer writes a private epoch progress checkpoint and resumes only if the manifest, feature/video hashes, configuration, PyTorch version and device match. If a final checkpoint exists without a report, preserve both before choosing new output paths.


In [20]:
# Validate all media/features and split provenance before training; inspect the private quality reports as well.
DATASET_VERIFIED = False
subprocess.run([PYTHON, '-m', 'deepfake_dtw.dataset', 'validate', str(PLAN)], cwd=REPO, env=env, check=True)
DATASET_VERIFIED = True


In [21]:
print("DATASET_VERIFIED:", DATASET_VERIFIED)
print("Feature files:", len(list(FEATURES.glob("*.npz"))))

DATASET_VERIFIED: True
Feature files: 181


In [23]:
assert globals().get('GPU_DIAGNOSTIC_PASSED', False) and globals().get('MODEL_SMOKE_PASSED', False) and globals().get('DATASET_VERIFIED', False), 'Complete GPU checks and dataset audit first'
checkpoint = PRIVATE / 'lavdf' / 'baseline.pt'
training_report = REPORTS / 'training.json'
training_progress = PRIVATE / 'lavdf' / 'baseline.progress.pt'
if checkpoint.exists() and training_report.exists():
    saved = json.loads(training_report.read_text())
    assert saved.get('checkpoint_sha256') == digest_file(checkpoint) and saved.get('manifest_sha256') == digest_file(PLAN), 'Existing training output does not match the current checkpoint or manifest; preserve it and choose new paths.'
    print('Reusing verified training output. Review validation results before final evaluation.')
else:
    assert not checkpoint.exists() and not training_report.exists(), 'Incomplete training output exists. Preserve and inspect it before moving it aside and retrying.'
    output = subprocess.check_output([PYTHON, '-m', 'deepfake_dtw.training', 'train', str(PLAN), str(checkpoint), '--seed', '42', '--epochs', '10', '--device', 'cuda', '--progress', str(training_progress)], cwd=REPO, env=env, text=True)
    report_data = json.loads(output)
    assert report_data['checkpoint_sha256'] == digest_file(checkpoint) and report_data['manifest_sha256'] == digest_file(PLAN)
    with training_report.open('x') as stream:
        json.dump(report_data, stream, indent=2)
    print('Training report saved privately; review validation before final evaluation.')


Reusing verified training output. Review validation results before final evaluation.


In [24]:
import json

report_path = REPORTS / "training.json"

with report_path.open() as f:
    training_report = json.load(f)

print(json.dumps(training_report, indent=2))

{
  "provenance_scope": "provider_source_recording",
  "identity_disjoint_verified": false,
  "best_epoch": 7,
  "validation_loss": 0.6022650599479675,
  "validation_selected_threshold": 0.49470940232276917,
  "training_clips": 60,
  "validation_clips": 60,
  "training_videos": 60,
  "validation_videos": 60,
  "dataset_origins": [
    "LAV-DF"
  ],
  "manifest_sha256": "de0aec15a4c33d93fc0a4c41c8c2c1be6f65abcb8a6e3a97ba9781819406229c",
  "checkpoint_sha256": "f2bbc8e03d963ea0ce077a68513c3328c9fc7b9e4b3c3ad4f184c283cad17902",
  "device": "cuda",
  "wall_seconds": 40.79802297199967,
  "process_peak_rss_kib_linux": 1125200,
  "note": "Validation loss only; no calibration or effectiveness claim."
}


In [25]:
assert globals().get('DATASET_VERIFIED', False), 'Complete the dataset audit first'
# Run only after reviewing validation and freezing all model choices.
report = REPORTS / 'held-out-test.json'
assert checkpoint.is_file(), 'Complete training first'
# Verify frozen artifact bindings without parsing held-out feature arrays.
verify_frozen = """import hashlib,json,sys,torch
with open(sys.argv[1], 'rb') as f: manifest_sha=hashlib.file_digest(f,'sha256').hexdigest()
with open(sys.argv[1]) as f: rows=json.load(f)
state=torch.load(sys.argv[2],map_location='cpu',weights_only=True)
assert state.get('format')=='deepfake_dtw_classifier_v2' and state['manifest_sha256']==manifest_sha
for row in rows:
    for path_key, hash_key in (('video_path','video_hashes'),('feature_path','feature_hashes')):
        with open(row[path_key],'rb') as f: value=hashlib.file_digest(f,'sha256').hexdigest()
        assert state[hash_key][row['sample_id']]==value, 'Frozen video or feature has changed'
"""
subprocess.run([PYTHON, '-c', verify_frozen, str(PLAN), str(checkpoint)], cwd=REPO, env=env, check=True)
if report.exists():
    saved = json.loads(report.read_text())
    assert saved.get('checkpoint_sha256') == digest_file(checkpoint) and saved.get('manifest_sha256') == digest_file(PLAN) and saved.get('partition') == 'test', 'Existing test report does not match this frozen experiment; preserve it and choose a new report path.'
    print('Reusing matching held-out report; no second evaluation was run.')
else:
    with report.open('x') as stream:
        subprocess.run([PYTHON, '-m', 'deepfake_dtw.training', 'evaluate', str(PLAN), str(checkpoint), '--partition', 'test', '--device', 'cuda'], cwd=REPO, env=env, stdout=stream, check=True)
    print('Held-out report saved privately.')


Held-out report saved privately.


In [26]:
import json

with (REPORTS / "held-out-test.json").open() as f:
    test_report = json.load(f)

print(json.dumps(test_report, indent=2))

{
  "accuracy": 0.7666666666666667,
  "precision": 0.7222222222222222,
  "recall": 0.8666666666666667,
  "f1": 0.7878787878787877,
  "roc_auc": 0.8255555555555556,
  "brier_score_uncalibrated": 0.17934844360799412,
  "false_positive_rate": 0.3333333333333333,
  "confusion_matrix": [
    [
      20,
      10
    ],
    [
      4,
      26
    ]
  ],
  "videos": 60,
  "provenance_scope": "provider_source_recording",
  "identity_disjoint_verified": false,
  "partition": "test",
  "threshold": 0.49470940232276917,
  "manifest_sha256": "de0aec15a4c33d93fc0a4c41c8c2c1be6f65abcb8a6e3a97ba9781819406229c",
  "checkpoint_sha256": "f2bbc8e03d963ea0ce077a68513c3328c9fc7b9e4b3c3ad4f184c283cad17902",
  "device": "cuda",
  "threshold_source": "validation video-level balanced accuracy; ties nearest 0.5",
  "class_counts": {
    "real": 30,
    "fake": 30
  },
  "false_positive_video_ids": [
    "test/090064.mp4",
    "test/041456.mp4",
    "test/091181.mp4",
    "test/041391.mp4",
    "test/002857.mp4